In [59]:
# import pdfplumber
# import fitz
from ingredient_parser import parse_ingredient
from recipe_scrapers import scrape_me, scrape_html
from urllib.request import urlopen
import re
import requests

### PDF Scraper Test

In [ ]:
# with pdfplumber.open("../recipes/mentaiko.pdf") as pdf:
#     full_text = ""
#     for page in pdf.pages:
#         words = page.extract_words(keep_blank_chars=True, x_tolerance=3, y_tolerance=3)
#         line = " ".join(word["text"] for word in words)
#         full_text += line + "\n"

In [25]:
doc = fitz.open("../recipes/mentaiko.pdf")
full_text = ""
for page in doc:
    text = page.get_text()
    full_text += text + "\n"

print(full_text)

Classic Mentaiko Pasta
by Namiko Hirasawa Chen
Japanese-style Mentaiko Pasta is one of the most popular wafu fusion dishes
in Japan. In this quick and easy recipe, we toss hot spaghetti in a buttery
cream sauce mixed with spicy salted cod or pollock roe. It‘s a heavenly,
delicious combination! Ready in just 20 minutes.
Prep: 5 mins
Cook: 15 mins
Total: 20 mins
Servings: 2
Ingredients
For the Toppings
Instructions
• 2 Tbsp Diamond Crystal kosher salt (for boiling the pasta)
• 8 oz spaghetti (4 oz, 113 g per serving)
• 2 sacs spicy salted pollock roe or cod roe (karashi mentaiko) (about 1 sac (1 oz, 30 g) per serving; I use
the kind without food coloring, so my dish does not look as pink/orange as other versions)
• 2 Tbsp whole milk
• 2 Tbsp heavy (whipping) cream
• 2 Tbsp unsalted butter (melted)
• 1 Tbsp soy sauce
• freshly ground black pepper
• shredded nori seaweed (or cut a nori sheet into thin strips 2 inches (5 cm) long)
• shiso leaves (perilla/ooba)
1. Gather all the ingredients.

In [26]:
full_text

'Classic Mentaiko Pasta\nby Namiko Hirasawa Chen\nJapanese-style Mentaiko Pasta is one of the most popular wafu fusion dishes\nin Japan. In this quick and easy recipe, we toss hot spaghetti in a buttery\ncream sauce mixed with spicy salted cod or pollock roe. It‘s a heavenly,\ndelicious combination! Ready in just 20 minutes.\nPrep: 5 mins\nCook: 15 mins\nTotal: 20 mins\nServings: 2\nIngredients\nFor the Toppings\nInstructions\n• 2 Tbsp Diamond Crystal kosher salt (for boiling the pasta)\n• 8 oz spaghetti (4 oz, 113 g per serving)\n• 2 sacs spicy salted pollock roe or cod roe (karashi mentaiko) (about 1 sac (1 oz, 30 g) per serving; I use\nthe kind without food coloring, so my dish does not look as pink/orange as other versions)\n• 2 Tbsp whole milk\n• 2 Tbsp heavy (whipping) cream\n• 2 Tbsp unsalted butter (melted)\n• 1 Tbsp soy sauce\n• freshly ground black pepper\n• shredded nori seaweed (or cut a nori sheet into thin strips 2 inches (5 cm) long)\n• shiso leaves (perilla/ooba)\n1. Ga

In [27]:
# Split into lines
lines = full_text.split("\n")

# Find sections
ingredients_raw = []
steps_raw = []
current_section = None

for line in lines:
    line = line.strip()
    if not line:
        continue

    # Detect section headers
    if re.search(r'\bingredients?\b', line, re.IGNORECASE):
        current_section = "ingredients"
        continue
    elif re.search(r'\b(instructions?|directions?|method|steps?)\b', line, re.IGNORECASE):
        current_section = "steps"
        continue

    # Add line to correct section
    if current_section == "ingredients":
        ingredients_raw.append(line)
    elif current_section == "steps":
        steps_raw.append(line)

# Parse ingredients
print("=== INGREDIENTS ===")
for line in ingredients_raw:
    try:
        parsed = parse_ingredient(line)
        print(parsed)
    except Exception:
        print(f"  (unparsed) {line}")

# Print steps
print("\n=== STEPS ===")
for i, step in enumerate(steps_raw, 1):
    print(f"{i}. {step}")

=== INGREDIENTS ===
ParsedIngredient(name=[IngredientText(text='For the Toppings', confidence=0.712792, starting_index=0)], size=None, amount=[], preparation=None, comment=None, purpose=None, foundation_foods=[], sentence='For the Toppings')
ParsedIngredient(name=[IngredientText(text='Diamond Crystal kosher salt', confidence=0.93411, starting_index=15)], size=None, amount=[IngredientAmount(quantity=Fraction(2, 1), quantity_max=Fraction(2, 1), unit='', text='2', confidence=0.996252, starting_index=0, unit_system=<UnitSystem.NONE: 'none'>, APPROXIMATE=False, SINGULAR=False, RANGE=False, MULTIPLIER=False, PREPARED_INGREDIENT=True), IngredientAmount(quantity=Fraction(2, 1), quantity_max=Fraction(2, 1), unit='Tbsps', text='2 Tbsps', confidence=0.749247, starting_index=13, unit_system=<UnitSystem.OTHER: 'other'>, APPROXIMATE=False, SINGULAR=False, RANGE=False, MULTIPLIER=False, PREPARED_INGREDIENT=False)], preparation=IngredientText(text='Bring a large pot of water to a boil., to the boiling

In [29]:
ingredients_raw

['For the Toppings',
 '2. Bring a large pot of water to a boil. Add 2 Tbsp Diamond Crystal kosher salt to the boiling water and',
 '7. When the spaghetti is done cooking, drain and transfer the pasta to the large bowl with the sauce.',
 '8. Toss to combine until the butter is melted and the sauce evenly coats the pasta. Taste and adjust the',
 'seasoning with black pepper and kosher salt.',
 'Classic Mentaiko Pasta - Just One Cookbook',
 'https://www.justonecookbook.com/wprm_print/classic-mentaiko-pasta',
 '3 of 4',
 '3/28/26, 4:54 PM',
 'Copyright © 2011–2026 Just One Cookbook®. All Rights Reserved.',
 'To Serve',
 'To Store',
 'Nutrition',
 'Calories: 664kcal, Carbohydrates: 88g, Protein: 23g, Fat: 24g, Saturated Fat: 12g, Polyunsaturated Fat: 3g,',
 'Monounsaturated Fat: 6g, Trans Fat: 1g, Cholesterol: 219mg, Sodium: 925mg, Potassium: 338mg, Fiber: 4g, Sugar: 4g,',
 'Vitamin A: 850IU, Vitamin C: 1mg, Calcium: 131mg, Iron: 5mg',
 '1. Serve onto individual plates. Roll up a stack of s

### Recipe Scraper

In [70]:
url = "https://www.mamaknowsglutenfree.com/gluten-free-chicken-parmesan/"
headers = {
    "User-Agent": "Mozilla/5.0 (iPhone; CPU iPhone OS 16_0 like Mac OS X) AppleWebKit/605.1.15"
}

response = requests.get(url, headers=headers)

html = response.text

In [71]:
scraper = scrape_html(html, url, wild_mode=True)
print(scraper.schema.data) 

{'@type': 'Recipe', 'name': 'Gluten-Free Chicken Parmesan', 'author': {'@id': 'https://www.mamaknowsglutenfree.com/#/schema/person/16fee4f9376d7857af73ce3e554026df'}, 'description': 'This gluten-free chicken parmesan gives me all the comfort. It’s pan-fried for that golden crunch, then finished in the oven with marinara and melty mozzarella. It’s surprisingly easy to pull together, whether it’s a busy weeknight or a slow Sunday dinner.', 'datePublished': '2025-09-01T08:30:00+00:00', 'image': ['https://www.mamaknowsglutenfree.com/wp-content/uploads/2025/06/1Gluten-Free-Chicken-Parmesan-18.jpg', 'https://www.mamaknowsglutenfree.com/wp-content/uploads/2025/06/1Gluten-Free-Chicken-Parmesan-18-500x500.jpg', 'https://www.mamaknowsglutenfree.com/wp-content/uploads/2025/06/1Gluten-Free-Chicken-Parmesan-18-500x375.jpg', 'https://www.mamaknowsglutenfree.com/wp-content/uploads/2025/06/1Gluten-Free-Chicken-Parmesan-18-480x270.jpg'], 'video': {'name': 'Gluten-Free Chicken Parmesan', 'description': 

/opt/homebrew/lib/python3.11/site-packages/recipe_scrapers/__init__.py:1233: DeprecationWarning: The 'wild_mode' parameter is deprecated and may be removed in future.

Please pass 'supported_only=False' instead for similar behaviour.
  warnings.warn(msg, category=DeprecationWarning)


In [72]:
print(scraper.title())
    
print(scraper.ingredients())

Gluten-Free Chicken Parmesan
['6 skinless, boneless thin-sliced chicken breasts (or skinless, boneless chicken breasts halved to make fillets)', 'salt and black pepper, to taste', '2 large eggs', '1 cup gluten-free panko bread crumbs (I like 4C gluten-free panko bread crumbs. I buy it at Walmart)', '1 cup gluten-free bread crumbs (or use 4C gluten-free bread crumbs)', '½ cup Parmesan cheese, grated (For dairy-free: I like Follow Your Heart dairy-free parmesan cheese)', '½ tsp garlic powder', '½ tsp onion powder', '1 tbsp Italian seasoning', '½ cup gluten-free flour (I like Pillsbury gluten-free flour)', '¼ cup cornstarch (Substitute for more gluten-free flour)', '½ cup olive oil', '½ cup gluten-free marina sauce', '1 cup mozzarella cheese, sliced or shredded (For dairy-free: I like Violife dairy-free mozzarella cheese shreds)', '½ cup Parmesan cheese', '¼ cup fresh basil (chopped)']


In [66]:
scraper.instructions()

'add ingredients to pot\nAdd all ingredients to a pot and heat over low/medium heat.\nwhisk together\nUse an immersion blender, milk frother, or whisk really well to ensure all of the spices are well combined.\nheat\nHeat until warm and steamy then mix really well again to get it nice and frothy. Taste and adjust flavors/sweetness as desired.\nserve\nTransfer to mugs and enjoy! The spices sometimes settle at the bottom of the mugs if you drink slowly, so feel free to strain it through a fine mesh strainer to avoid that!'

In [69]:
url = "https://www.mamaknowsglutenfree.com/gluten-free-chicken-parmesan/"
scraper = scrape_me(url)
print(scraper.ingredients())
print(scraper.instructions())

WebsiteNotImplementedError: recipe-scrapers (15.11.0) exception: Website (The website 'mamaknowsglutenfree.com' isn't currently supported by recipe-scrapers!
---
If you have time to help us out, please report this as a feature 
request on our bugtracker.) not supported.

In [68]:
for ingredient in scraper.ingredients():
    print(ingredient.strip())

2 tablespoons grated and peeled fresh ginger
1 teaspoon sugar
1 teaspoon coarse salt
2 cups jasmine rice


In [49]:
CUSTOM_UNITS = [
    "sac", "sacs", "bunch", "bunches", "sprig", "sprigs",
    "fillet", "fillets", "slice", "slices", "piece", "pieces",
    "stalk", "stalks", "head", "heads", "clove", "cloves",
    "sheet", "sheets", "block", "blocks", "can", "cans",
    "jar", "jars", "bag", "bags", "package", "packages", "cup",
]

In [50]:
def format_ingredient(raw_string):
    parsed = parse_ingredient(raw_string)
    
    amount = None
    unit = None
    item = raw_string  # fallback to raw string

    if parsed.amount:
        amount = parsed.amount[0].quantity
        unit = parsed.amount[0].unit

    if parsed.name:
        item = parsed.name[0].text

    # Check if unit was misclassified as part of item name
    if item:
        words = item.split()
        if words[0].lower() in CUSTOM_UNITS:
            unit = words[0]
            item = " ".join(words[1:])

    return {
        "amount": amount,
        "unit": unit,
        "item": item
    }

ingredients = [format_ingredient(i) for i in scraper.ingredients()]

In [51]:
ingredients

[{'amount': Fraction(1, 1),
  'unit': <Unit('cup')>,
  'item': 'all-purpose flour (plain flour)'},
 {'amount': Fraction(1, 4),
  'unit': <Unit('teaspoon')>,
  'item': 'Diamond Crystal kosher salt'},
 {'amount': Fraction(1, 4), 'unit': <Unit('teaspoon')>, 'item': 'sugar'},
 {'amount': Fraction(1, 4),
  'unit': <Unit('teaspoon')>,
  'item': 'baking powder'},
 {'amount': Fraction(28, 5),
  'unit': <Unit('ounce')>,
  'item': 'nagaimo/yamaimo'},
 {'amount': Fraction(3, 4), 'unit': <Unit('cup')>, 'item': 'dashi'},
 {'amount': Fraction(1, 2), 'unit': 'heads', 'item': 'green cabbage'},
 {'amount': Fraction(1, 4),
  'unit': <Unit('cup')>,
  'item': 'pickled red ginger'},
 {'amount': Fraction(1, 2), 'unit': <Unit('pound')>, 'item': 'pork belly'},
 {'amount': Fraction(4, 1), 'unit': '', 'item': 'eggs'},
 {'amount': Fraction(1, 2),
  'unit': <Unit('cup')>,
  'item': 'tenkasu/agedama (tempura scraps)'},
 {'amount': None, 'unit': None, 'item': 'neutral oil'},
 {'amount': None, 'unit': None, 'item': 

In [45]:
steps_raw = scraper.instructions().split("\n")
steps_clean = [step.strip() for step in steps_raw if step.strip()]
instructions = []

for i, step in enumerate(steps_clean, 1):
    instructions.append(f"{i}. {step}")

In [46]:
instructions

['1. Gather all the ingredients.',
 '2. Bring a large pot of water to a boil. Add 2 Tbsp Diamond Crystal kosher salt to the boiling water and cook 8 oz spaghetti until al dente, about 10 minutes (or according to the package instructions).',
 '3. In a large bowl, combine 2 Tbsp whole milk, 2 Tbsp heavy (whipping) cream, 2 Tbsp unsalted butter, and 1 Tbsp soy sauce. Don‘t worry if the butter solidifies. The hot spaghetti will melt the butter later.',
 '4. Add freshly ground black pepper and stir to combine.',
 '5. Make a lengthwise slit in the membranes of 2 sacs spicy salted pollock roe or cod roe (karashi mentaiko) to open. Squeeze out the roe from the sacs with your hands or a knife. Discard the membrane.',
 '6. Add the roe to the bowl with the sauce ingredients and stir well.',
 '7. When the spaghetti is done cooking, drain and transfer the pasta to the large bowl with the sauce.',
 '8. Toss to combine until the butter is melted and the sauce evenly coats the pasta. Taste and adjust 

In [ ]:
output = {
  "url": url,
  "name": None,
  "image": None,
  "servings": None,
  "total_time": None,
  "ingredients": ingredients,
  "instructions": instructions
}